## Understanding Regrid Rate Limits

**IMPORTANT:** Regrid API has per-plan rate limits. Exceeding them returns HTTP 429 "Too Many Requests" errors.

### Rate Limits by Regrid Plan

| Plan | Rate Limit | Recommended `MIN_INTERVAL_SECS` | Time for 1M requests |
|------|-----------|--------------------------------|----------------------|
| Starter | ~1 req/sec | 1.0 | ~11.5 days |
| Growth | ~5 req/sec | 0.2 | ~2.3 days |
| Professional | ~10 req/sec | 0.1 | ~1.2 days |
| Enterprise | Negotiated | As low as 0.05 or better | Hours to days |

### If You See "Rate Limited (429)" Errors

1. **Increase `MIN_INTERVAL_SECS`** in the configuration cell
   - Current: check the printed rate after running the cell
   - Increase by 50%: if `0.2`, try `0.3`
   - Restart kernel and re-run

2. **For production (1M+ prospects)**, use the command-line pipeline with parallel workers instead

### How Rate Limiting Works in This Notebook

- **`min_interval`** — Enforced globally across all requests (thread-safe lock ensures spacing)
- **`max_retries`** — Automatically retries on transient errors (including 429) with exponential backoff
- **`backoff_base`** — Each retry waits longer: 1.5^attempt + random jitter

### Next: Switch to Real Backend

Once validated with mock data, switch to the real Regrid API:

#### ✓ Validation Checklist (Mock Mode)

- [ ] No errors in mock run
- [ ] Classification logic working (reasonable truck_flag distribution)
- [ ] Output CSV format correct (all fields present)
- [ ] Incremental resumability works (test interrupt + re-run)

#### 🔄 Switching to Real Mode

1. **Identify your Regrid plan tier** — Check your dashboard or docs for your rate limit

2. **Set `MIN_INTERVAL_SECS` conservatively** — Start with a value slightly higher than your plan's rate:
   - Growth (5/sec) → Start with 0.3 seconds
   - Professional (10/sec) → Start with 0.15 seconds
   - Enterprise → Start with 0.1 seconds

3. **Set your Regrid API URL**:
   ```bash
   export REGRID_QUERY_URL="https://fs.regrid.com/<your-token>/rest/services/premium/FeatureServer/0/query"
   ```

4. **Change notebook flags** in the configuration cell:
   ```python
   USE_MOCK = False  # Use live API
   MIN_INTERVAL_SECS = 0.3  # Adjust for your plan
   LIMIT_ROWS = 100  # Start small to test
   ```

5. **Run notebook** — Watch for rate-limit errors (429 responses)

6. **Monitor the progress line**: `n_done | flagged | errors | rate_limited | X.XX rows/sec | ETA`
   - If `rate_limited` count keeps growing → increase `MIN_INTERVAL_SECS`
   - If rows/sec is much lower than expected → decrease `MIN_INTERVAL_SECS` (if no errors)

In [ ]:
# --- Validate and summarize results ---

# Load results
result_df = pd.read_csv(output_file)
print(f"✓ Loaded results: {len(result_df)} rows")

# Summary statistics
print("\n📈 Results Summary:")
print(f"  Total processed: {len(result_df)}")
print(f"  Flagged (truck): {(result_df['truck_flag'] == 'True').sum()}")
print(f"  High confidence: {(result_df['confidence'] == 'high').sum()}")
print(f"  Medium confidence: {(result_df['confidence'] == 'medium').sum()}")
print(f"  Low confidence: {(result_df['confidence'] == 'low').sum()}")
print(f"  Errors: {(result_df['error'] != '').sum()}")

print("\n📊 Sample flagged records (truck_flag=True):")
flagged = result_df[result_df['truck_flag'] == 'True']
if len(flagged) > 0:
    print(flagged[['point_id', 'lat', 'lon', 'confidence', 'parcel_address', 'county']].head(5).to_string())
else:
    print("  (no truck flags in this batch)")

print("\n📊 Sample errors (if any):")
errors = result_df[result_df['error'] != '']
if len(errors) > 0:
    print(errors[['point_id', 'error']].head(3).to_string())
else:
    print("  (no errors)")

In [ ]:
# --- Run the enrichment pipeline with rate limit tracking ---

output_file = os.path.join(os.getcwd(), "enriched_prospects.csv")

# Check if we can resume
already_done = set()
if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
    with open(output_file, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("point_id"):
                already_done.add(row["point_id"])

if already_done:
    print(f"[pipeline] Resuming: {len(already_done)} rows already in {output_file}, skipping them")

# Filter to-do rows
todo = df[~df["point_id"].astype(str).isin(already_done)]
total = len(todo)
print(f"[pipeline] {total} rows to process (of {len(df)} total in input)")

if total == 0:
    print("[pipeline] Nothing to do.")
else:
    # Write header if new file
    write_header = not os.path.exists(output_file) or os.path.getsize(output_file) == 0
    out_f = open(output_file, "a", newline="", encoding="utf-8")
    writer = csv.DictWriter(out_f, fieldnames=OUTPUT_FIELDS)
    if write_header:
        writer.writeheader()
        out_f.flush()

    start = time.monotonic()
    n_done = 0
    n_flagged = 0
    n_error = 0
    n_rate_limited = 0
    rate_limited_pids = []

    try:
        # Sequential processing for notebook (use ThreadPoolExecutor for parallel in production)
        for _, row in todo.iterrows():
            try:
                row_out = process_one(
                    client, 
                    row["point_id"], 
                    float(row["lat"]), 
                    float(row["lon"])
                )
            except Exception as exc:
                # Catch rate limit errors explicitly
                if "429" in str(exc) or "rate limited" in str(exc).lower():
                    n_rate_limited += 1
                    rate_limited_pids.append(str(row["point_id"]))
                    print(f"\n⚠️  RATE LIMITED (429) at row {n_done + 1}. Check your min_interval setting.")
                    print(f"    Error: {exc}")
                    print(f"    Current rate limit: {1/MIN_INTERVAL_SECS:.2f} req/sec")
                    print(f"    Try increasing MIN_INTERVAL_SECS to reduce request rate.\n")
                    # Create error row
                    row_out = {
                        "point_id": str(row["point_id"]), 
                        "lat": float(row["lat"]), 
                        "lon": float(row["lon"]),
                        "truck_flag": "", "confidence": "", "reasons": "",
                        "parcel_owner": "", "parcel_address": "", 
                        "parcel_zoning_type": "", "parcel_zoning_subtype": "", 
                        "county": "", "state": "",
                        "error": f"RateLimitError: {exc}",
                    }
                else:
                    raise
            
            writer.writerow(row_out)
            n_done += 1
            if row_out["truck_flag"] == "True" or row_out["truck_flag"] is True:
                n_flagged += 1
            if row_out["error"]:
                n_error += 1
            
            if n_done % 10 == 0 or n_done == total:
                out_f.flush()
                elapsed = time.monotonic() - start
                rate = n_done / elapsed if elapsed > 0 else 0
                eta = (total - n_done) / rate if rate > 0 else float("inf")
                print(f"[pipeline] {n_done}/{total} done | flagged={n_flagged} errors={n_error} "
                      f"rate_limited={n_rate_limited} | {rate:.2f} rows/sec | ETA {eta/60:.1f} min")
    finally:
        out_f.flush()
        out_f.close()

    print(f"\n[pipeline] Finished. {n_done} rows processed, {n_flagged} flagged, {n_error} errors, {n_rate_limited} rate-limited.")
    print(f"[pipeline] Output: {output_file}")
    
    if n_rate_limited > 0:
        print(f"\n⚠️  Rate limit was hit {n_rate_limited} times.")
        print(f"    If this is frequent, INCREASE MIN_INTERVAL_SECS in the configuration cell.")
        print(f"    Current setting: {MIN_INTERVAL_SECS} sec/request = {1/MIN_INTERVAL_SECS:.2f} req/sec")

In [ ]:
# --- Define pipeline helper function ---

OUTPUT_FIELDS = [
    "point_id", "lat", "lon", "truck_flag", "confidence", "reasons",
    "parcel_owner", "parcel_address", "parcel_zoning_type", "parcel_zoning_subtype",
    "county", "state", "error",
]

def process_one(client, point_id, lat, lon):
    """Process a single prospect point through the Regrid pipeline."""
    try:
        attrs = client.query_point(lat, lon)
    except Exception as exc:
        return {
            "point_id": str(point_id), "lat": lat, "lon": lon,
            "truck_flag": "", "confidence": "", "reasons": "",
            "parcel_owner": "", "parcel_address": "", "parcel_zoning_type": "",
            "parcel_zoning_subtype": "", "county": "", "state": "",
            "error": f"{type(exc).__name__}: {exc}",
        }

    if attrs is None:
        result = classify_parcel({})
        return {
            "point_id": str(point_id), "lat": lat, "lon": lon,
            "truck_flag": result["truck_flag"], "confidence": result["confidence"],
            "reasons": "no parcel found; " + "; ".join(result["reasons"]),
            "parcel_owner": "", "parcel_address": "", "parcel_zoning_type": "",
            "parcel_zoning_subtype": "", "county": "", "state": "",
            "error": "",
        }

    result = classify_parcel(attrs)
    return {
        "point_id": str(point_id), "lat": lat, "lon": lon,
        "truck_flag": result["truck_flag"], "confidence": result["confidence"],
        "reasons": "; ".join(result["reasons"]),
        "parcel_owner": attrs.get("owner") or "",
        "parcel_address": attrs.get("address") or "",
        "parcel_zoning_type": attrs.get("zoning_type") or "",
        "parcel_zoning_subtype": attrs.get("zoning_subtype") or "",
        "county": attrs.get("county") or "",
        "state": attrs.get("state2") or "",
        "error": "",
    }

print("✓ Pipeline helper function defined")

In [ ]:
# --- Inspect schema and sample records ---

print("📊 Input Schema:")
print(df.dtypes)
print(f"\n📊 Sample records (first 3 rows):")
print(df.head(3).to_string())

print(f"\n📈 Basic Statistics:")
print(f"  Total rows: {len(df)}")
print(f"  Latitude range: {df['lat'].min():.4f} to {df['lat'].max():.4f}")
print(f"  Longitude range: {df['lon'].min():.4f} to {df['lon'].max():.4f}")

In [ ]:
# --- Load input prospects ---
# Look for mock_prospect_points.parquet in project root or fixtures

possible_paths = [
    Path(project_root) / "mock_prospect_points.parquet",
    Path(project_root) / "tests" / "fixtures" / "mock_prospect_points.parquet",
    Path(project_root) / "data" / "mock_prospect_points.parquet",
]

input_file = None
for p in possible_paths:
    if p.exists():
        input_file = str(p)
        break

if input_file is None:
    print("⚠️  Mock prospect file not found in expected locations:")
    for p in possible_paths:
        print(f"    - {p}")
    print("\nPlease copy mock_prospect_points.parquet from truck_parcel_pipeline project")
    print(f"or specify the path manually.")
    raise FileNotFoundError("mock_prospect_points.parquet not found")

df = pd.read_parquet(input_file)
print(f"✓ Loaded {len(df)} prospect points from {input_file}")
print(f"  Columns: {list(df.columns)}")

# Limit rows for testing
if LIMIT_ROWS:
    df = df.head(LIMIT_ROWS)
    print(f"✓ Limited to {LIMIT_ROWS} rows for testing")

df.head(3)

In [ ]:
# --- Configuration: Switch between mock and live modes ---
# Rate Limit Settings (adjust based on your Regrid service tier)
# Regrid API limits vary by plan:
#   - Starter: ~1 req/sec (min_interval=1.0)
#   - Growth: ~5 req/sec (min_interval=0.2)
#   - Professional: ~10 req/sec (min_interval=0.1)
#   - Enterprise: negotiated (typically 0.05 or better)
# If you see 429 "Too Many Requests" errors, INCREASE min_interval

USE_MOCK = True  # Set to False to use live Regrid API
LIMIT_ROWS = 100  # Limit rows for testing (set to None for full run)

# Rate limiting configuration (only used in live mode)
MIN_INTERVAL_SECS = 0.2  # Start conservative; adjust based on your Regrid plan
MAX_RETRIES = 4  # Retry attempts before giving up
BACKOFF_BASE = 1.5  # Exponential backoff multiplier (1.5^attempt + random jitter)

if USE_MOCK:
    from src.regrid_client import MockRegridClient
    client = MockRegridClient()
    print("[config] Using MockRegridClient (no network calls)")
else:
    from src.regrid_client import LiveRegridClient
    query_url = os.environ.get("REGRID_QUERY_URL")
    if not query_url:
        raise ValueError("Set REGRID_QUERY_URL environment variable for live mode")
    client = LiveRegridClient(
        query_url, 
        min_interval=MIN_INTERVAL_SECS,
        max_retries=MAX_RETRIES,
        backoff_base=BACKOFF_BASE,
        timeout=15
    )
    print("[config] Using LiveRegridClient (real Regrid API)")
    print(f"[config] Rate limit: {1/MIN_INTERVAL_SECS:.2f} requests/second")
    print(f"[config] Max retries: {MAX_RETRIES} with {BACKOFF_BASE}x backoff")

print(f"[config] Mock mode: {USE_MOCK}")
print(f"[config] Row limit: {LIMIT_ROWS}")

In [ ]:
# Import required libraries
import pandas as pd
import pyarrow.parquet as pq
import csv
import os
import time
from pathlib import Path

from src.classify import classify_parcel

In [ ]:
# Setup: Make src modules importable
import os, sys
from pathlib import Path

# Add parent (project root) to path so we can import from src
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(notebook_dir))

# Regrid Parcel Enrichment - Spatial Analysis

Flag prospect locations as truck/truck-parking candidates using Regrid parcel data.

This notebook runs locally with mock fixtures for quick iteration, or connects to the live Regrid API for enriching real prospect data.

Start on the **local fixtures** (mock data, no network calls); flip the flags to hit the real Regrid FeatureServer API once the pipeline looks right.

## What this does

1. **Loads prospect locations** — Parquet file with (id, lat, lon) columns
2. **Classifies parcels** — Uses Regrid API to look up parcels, classifies them as truck/logistics-likely
3. **Incremental output** — Writes CSV incrementally (resumable if interrupted)
4. **Mock mode** — Test pipeline with synthetic data before running against real prospects

## Prerequisites

- Modules: `classify`, `regrid_client`, `real_fixtures` (in `src/`)
- Mock fixtures: parquet file with prospect points  
- Real mode: `REGRID_QUERY_URL` environment variable with your Regrid FeatureServer token
- Optional AWS: S3 access if reading input from S3